# Chapitre 13 - Exercices 13.5 à 13.8

* **13.5** : meilleure approximation de rang $L$ d'une famille de matrices (réduction de
  dimension globale, éq. (13.11)).
* **13.6** : moyenne de formes invariante par transformation affine.
* **13.7** : k-ppv sur les problèmes « facile » et « difficile » ; validation croisée contre
  pénalisation de type AIC.
* **13.8** : invariance par rotation : k-ppv, « hints », métrique invariante, distance
  tangente.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import i0e
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch13_prototypes as cp
set_style()
PURPLE = COLORS[6]

## Exercice 13.5 : Meilleure approximation de rang $L$

**Énoncé (reformulé).** Soient $B_1, \dots, B_N$ des matrices $p \times p$ symétriques
semi-définies positives et $\bar B = \frac1N\sum_i B_i$, de décomposition spectrale
$\bar B = \sum_{\ell=1}^p \theta_\ell e_\ell e_\ell^T$ avec $\theta_1 \ge \theta_2 \ge \dots \ge \theta_p$.
Montrer que la meilleure approximation de rang $L$ des $B_i$,
$$\min_{\operatorname{rang}(M) = L} \ \sum_{i=1}^N \operatorname{trace}\big[(B_i - M)^2\big],$$
est donnée par $\bar B_{[L]} = \sum_{\ell=1}^L \theta_\ell e_\ell e_\ell^T$.
*Indication* : écrire $\sum_i \operatorname{trace}[(B_i - M)^2]$ comme
$\sum_i \operatorname{trace}[(B_i - \bar B)^2] + \sum_i \operatorname{trace}[(M - \bar B)^2]$.

(Le livre ordonne les valeurs propres avec une coquille d'indice ; nous les prenons
décroissantes.)

### Rappels

* **Produit scalaire de Frobenius** : $\langle A, C\rangle = \operatorname{trace}(A^TC)$,
  de norme $\|A\|_F^2 = \sum_{jk} A_{jk}^2$. Pour $A$ **symétrique**,
  $\operatorname{trace}(A^2) = \operatorname{trace}(A^TA) = \|A\|_F^2$. Nous cherchons donc
  $M$ symétrique (c'est naturel, et c'est nécessaire pour que le critère soit une somme de
  carrés).
* **Théorème spectral** : une matrice symétrique réelle se diagonalise dans une base
  orthonormée de vecteurs propres.
* La trace est linéaire et **invariante par changement de base orthonormée** :
  $\operatorname{trace}(U^TAU) = \operatorname{trace}(A)$ si $U$ est orthogonale.

### Étape 1 : décomposition « biais–variance » (l'indication)

On écrit $B_i - M = (B_i - \bar B) + (\bar B - M)$ et on développe le carré (pour des
matrices symétriques, $\operatorname{trace}((X + Y)^2) = \operatorname{trace}(X^2) + 2\operatorname{trace}(XY) + \operatorname{trace}(Y^2)$,
car $\operatorname{trace}(XY) = \operatorname{trace}(YX)$) :
$$\sum_i \operatorname{trace}[(B_i - M)^2] = \sum_i \operatorname{trace}[(B_i - \bar B)^2]
+ 2\operatorname{trace}\Big[\Big(\sum_i (B_i - \bar B)\Big)(\bar B - M)\Big] + N\operatorname{trace}[(\bar B - M)^2].$$
Par définition de la moyenne, $\sum_i (B_i - \bar B) = 0$ : le terme croisé disparaît.
Le premier terme ne dépend pas de $M$. Le problème revient donc à
$$\min_{\operatorname{rang}(M) \le L} \|\bar B - M\|_F^2 .$$

### Étape 2 : un lemme sur les traces (Ky Fan)

**Lemme.** Soit $C$ symétrique de valeurs propres $\lambda_1 \ge \dots \ge \lambda_p$
(vecteurs propres orthonormés $e_j$), et $q_1, \dots, q_L$ des vecteurs orthonormés. Alors
$\sum_{i=1}^L q_i^T C q_i \le \lambda_1 + \dots + \lambda_L$.

*Preuve.* Décomposons dans la base propre : $q_i^T C q_i = \sum_j \lambda_j (e_j^T q_i)^2$.
Donc $\sum_i q_i^TCq_i = \sum_j \lambda_j w_j$ avec $w_j = \sum_i (e_j^Tq_i)^2$.
* $w_j \le 1$ : c'est le carré de la norme de la projection de $e_j$ sur
  $\operatorname{vect}(q_1, \dots, q_L)$, inférieur à $\|e_j\|^2 = 1$ ;
* $\sum_j w_j = \sum_i \|q_i\|^2 = L$ (Pythagore dans la base $e_j$).

Maximiser $\sum_j \lambda_j w_j$ sous $0 \le w_j \le 1$, $\sum_j w_j = L$ revient à mettre
tout le poids possible sur les plus grandes valeurs : le maximum vaut
$\lambda_1 + \dots + \lambda_L$ (prendre $w_1 = \dots = w_L = 1$). $\square$

### Étape 3 : minoration et cas d'égalité

Soit $M$ de rang $\le L$ et $Q$ le projecteur orthogonal sur un sous-espace de dimension
$L$ contenant l'image de $M$ ; $Q = \sum_{i=1}^L q_iq_i^T$ avec des $q_i$ orthonormés, et
$(I - Q)M = 0$. Par Pythagore (les colonnes de $Q X$ et $(I-Q)X$ sont orthogonales),
$$\|\bar B - M\|_F^2 \ge \|(I - Q)(\bar B - M)\|_F^2 = \|(I - Q)\bar B\|_F^2
= \operatorname{trace}(\bar B^2) - \operatorname{trace}(Q \bar B^2 Q),$$
en développant $\operatorname{trace}(\bar B(I-Q)^2\bar B)$ avec $(I-Q)^2 = I - Q$. Or
$\operatorname{trace}(Q\bar B^2Q) = \sum_i q_i^T \bar B^2 q_i \le \theta_1^2 + \dots + \theta_L^2$
par le lemme appliqué à $C = \bar B^2$, dont les valeurs propres sont les $\theta_\ell^2$,
**dans le même ordre** car $\theta_\ell \ge 0$ ($\bar B$ est semi-définie positive, comme
moyenne de matrices qui le sont). D'où, avec $\operatorname{trace}(\bar B^2) = \sum_\ell \theta_\ell^2$,
$$\|\bar B - M\|_F^2 \ge \sum_{\ell > L} \theta_\ell^2 .$$
Enfin, $\bar B - \bar B_{[L]} = \sum_{\ell > L}\theta_\ell e_\ell e_\ell^T$ a exactement pour norme
au carré $\sum_{\ell > L}\theta_\ell^2$ : la borne est atteinte par $M = \bar B_{[L]}$. $\square$

### Vérification numérique

On tire $N = 30$ matrices $B_i$ semi-définies positives de rang 2 (comme des matrices
inter-classes locales), et on compare le critère en $\bar B_{[L]}$ à celui obtenu par
optimisation numérique directe sur les matrices symétriques de rang $L$
($M = U\operatorname{diag}(d)U^T$, $U$ de taille $p \times L$), depuis 20 points de départ.

In [2]:
rng = np.random.RandomState(0)
p, N, L = 6, 30, 2
Bs = []
for _ in range(N):
    G = rng.randn(p, 2) * np.array([3, 2, 1, 0.5, 0.3, 0.2])[:, None]
    Bs.append(G @ G.T)
Bs = np.array(Bs)
Bbar = Bs.mean(0)


def crit(M):
    return sum(np.trace((B - M) @ (B - M)) for B in Bs)


theta, E = np.linalg.eigh(Bbar)
theta, E = theta[::-1], E[:, ::-1]
B_L = (E[:, :L] * theta[:L]) @ E[:, :L].T
best = crit(B_L)


def obj(v):
    U = v[:p * L].reshape(p, L)
    return crit((U * v[p * L:]) @ U.T)


found = [minimize(obj, rng.randn(p * L + L), method="BFGS").fun for _ in range(20)]
print("critère en B̄_[L]                  : %.4f" % best)
print("meilleur critère trouvé numériquement : %.4f" % min(found))
print("formule  N*sum_{l>L} theta_l^2 + sum_i ||B_i - B̄||² : %.4f"
      % (N * (theta[L:] ** 2).sum() + sum(np.trace((B - Bbar) @ (B - Bbar)) for B in Bs)))

critère en B̄_[L]                  : 18035.5601
meilleur critère trouvé numériquement : 18035.5601
formule  N*sum_{l>L} theta_l^2 + sum_i ||B_i - B̄||² : 18035.5601
